# Agent Test Notebook

Use this notebook to test each individual agent in isolation. Set the agent name, query, time range, data products, and optional context, then run the execution cell.

The notebook supports the following agents: `information_agent`, `knowledge_agent`, `metadata_agent`, `capacity_agent`, `rule_agent`, and `supervisor_agent`.

import os
import sys
from pprint import pprint

# Ensure repo root is on sys.path when running from the notebook root
repo_root = os.path.abspath(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from core.base_agent import AgentRequest
from agents.information_agent import InformationAgent
from agents.knowledge_agent import KnowledgeAgent
from agents.metadata_agent import MetadataAgent
from agents.capacity_agent import CapacityAgent
from agents.rule_agent import RuleAgent
from agents.supervisor_agent import SupervisorAgent

AGENT_FACTORIES = {
    'information_agent': InformationAgent,
    'knowledge_agent': KnowledgeAgent,
    'metadata_agent': MetadataAgent,
    'capacity_agent': CapacityAgent,
    'rule_agent': RuleAgent,
    'supervisor_agent': SupervisorAgent,
}

AGENT_DESCRIPTIONS = {
    'information_agent': 'Returns structured metrics and anomaly signals.',
    'knowledge_agent': 'Returns business knowledge, definitions, and context.',
    'metadata_agent': 'Returns governance metadata and data quality details.',
    'capacity_agent': 'Returns Jira issue status or creates a ticket for a write query.',
    'rule_agent': 'Lists, creates, or evaluates rules based on your query.',
    'supervisor_agent': 'Runs the supervisor logic and executes all agents together.',
}

def get_agent(agent_name: str, enable_mock: bool = True):
    cls = AGENT_FACTORIES.get(agent_name)
    if cls is None:
        raise ValueError(f'Unknown agent: {agent_name}')
    return cls(enable_mock=enable_mock)

def run_agent(agent_name: str, query: str, time_range: str = 'last_month',
              data_products=None, context=None, enable_mock: bool = True):
    agent = get_agent(agent_name, enable_mock=enable_mock)
    request = AgentRequest(
        query=query,
        time_range=time_range,
        data_products=data_products or [],
        context=context or {},
    )
    return agent.execute(request)

def show_agent_details():
    print('Available agents:')
    for name, desc in AGENT_DESCRIPTIONS.items():
        print(f' - {name}: {desc}')

show_agent_details()

# Notebook control variables
selected_agent = 'information_agent'  # change this to any supported agent
query = 'Why did retention drop in the last month?'
time_range = 'last_month'
data_products = ['retention']  # set product tags if you want them explicitly
context = {
    # Optional context values for write operations, ticket creation, or rule creation
    # 'ticket_summary': 'Create a ticket for missing retention pipeline data',
    # 'ticket_description': 'The retention ETL failed for 3 days in the EU region.',
    # 'rule_name': 'Retention threshold rule',
    # 'expression': 'gross_retention_rate >= 85',
    # 'threshold': 85,
}
enable_mock = True  # set False only if your production connectors are configured

print(f'Selected agent: {selected_agent}')
print(f'Query: {query}')
print(f'Time range: {time_range}')
print(f'Data products: {data_products}')
print(f'Enable mock: {enable_mock}')

# Execute the selected agent and inspect the result
result = run_agent(
    selected_agent,
    query,
    time_range=time_range,
    data_products=data_products,
    context=context,
    enable_mock=enable_mock,
)

print('---')
print(f'Agent: {result.agent_name}')
print(f'Success: {result.success}')
print(f'Confidence: {result.confidence}')
print(f'Summary:
{result.summary}')
print('---')
print('Sources:', result.sources)
print('Metadata:', result.metadata)
print('Execution time (ms):', result.execution_time_ms)

print('Data payload:')
pprint(result.data)

## Example agent test scenarios

1. `information_agent`: query `Why did retention drop?` and `data_products=['retention']`.
2. `knowledge_agent`: query `What is GRR and why does it matter?`.
3. `metadata_agent`: query `Show governance metadata for retention`.
4. `capacity_agent`: query `List open issues for retention` or `Create ticket for missing data`.
5. `rule_agent`: query `List rules` or `Evaluate retention rules`.
6. `supervisor_agent`: query `Why did retention drop?` to run the full supervisor flow.

Change the variables in the previous cell and re-run the execution cell.